# Stage 2.1 - Fold-isolated FCMAE P1 and bidirectional cross-view P2

This notebook implements the authoritative `baseline_protocol_v1` encoder foundation.

- It consumes only the certified 71-knee manifest and never opens test-fold files.
- P1 uses canonical uniform 60% FCMAE masking; no target, fracture label, cohort sampler, or structure-derived mask enters pretraining.
- P2 adds bidirectional AP/LAT completion while retaining the FCMAE loss.
- Training augmentation is deterministic and photometric only. Validation is always clean.
- Every fold starts independently from the pinned self-supervised `convnextv2_tiny.fcmae` initialization.

`RUN_REAL_DATA` is deliberately `False` by default. A real run must be started only from an approved Stage 2 task brief after Agent N records Stage 1 `PASS`.

In [ ]:
import contextlib
import hashlib
import json
import math
import os
try:
    import resource
except ImportError:
    resource = None
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler, get_worker_info
import timm

# Environment/configuration cell. Algorithm cells are identical in the local and HPC twins.
PROTOCOL_VERSION = "baseline_protocol_v1"
STAGE2_SCHEMA = "foundation_stage2_v1"
SEED = 42
FOLD = 0
RUN_REAL_DATA = False
RUN_PREFLIGHT = False
RUN_P1 = False
RUN_P2 = False
RESUME_P1 = None
RESUME_P2 = None

PRETRAIN_MODEL = "convnextv2_tiny.fcmae"
IMAGE_SIZE = 256
PATCH_SIZE = 32
MASK_RATIO = 0.60
DECODER_DIM = 512
XVIEW_WEIGHT = 1.0
XVIEW_ROW_SLACK = 1
FEATURE_STD_MIN = 1e-6

P1_MAX_EPOCHS = 250
P2_MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 30
MICRO_BATCH = 16
ACCUM_STEPS = 8
EFFECTIVE_BATCH = 128
BASE_LR = 1.5e-4
PEAK_LR = BASE_LR * EFFECTIVE_BATCH / 256
WARMUP_FRACTION = 0.08
WEIGHT_DECAY = 0.05
NUM_WORKERS = 4 if os.name != "nt" else 0
USE_AMP = True
CHECKPOINT_EVERY = 25

AUGMENTATION = {
    "kind": "online_photometric_only",
    "gamma": [0.90, 1.10],
    "brightness": [-0.05, 0.05],
    "gaussian_noise_sigma": [0.0, 0.02],
    "clamp": [0.0, 1.0],
    "independent_ap_lat": True,
    "forbidden": ["crop", "rotation", "translation", "flip", "elastic", "cutout", "random_erasing"],
}
OOM_POLICY = {
    "action": "restart_from_pinned_initialization_with_half_micro_batch_and_double_accumulation",
    "effective_batch_must_remain": EFFECTIVE_BATCH,
}

assert 0 <= FOLD < 5
assert MICRO_BATCH * ACCUM_STEPS == EFFECTIVE_BATCH
assert IMAGE_SIZE % PATCH_SIZE == 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"fold": FOLD, "device": str(DEVICE), "run_real_data": RUN_REAL_DATA, "peak_lr": PEAK_LR})

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "configs" / "baseline_protocol_v1.json").exists() and (candidate / "reports" / "manifests").exists():
            return candidate
    raise FileNotFoundError("Project root with configs/ and reports/manifests/ was not found")


ROOT = find_project_root(Path.cwd())
MANIFEST_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.csv"
MANIFEST_META_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.metadata.json"
BASELINE_CONFIG_PATH = ROOT / "configs" / "baseline_protocol_v1.json"
DATA_CONFIG_PATH = ROOT / "configs" / "data_contract_v1.json"
ARTIFACT_ROOT = ROOT / "models" / STAGE2_SCHEMA / f"fold_{FOLD}"
STAGE1_GATE_PATH = ROOT / "reports" / "agent_runs" / "stage1" / "s1_manifest_certification_v1" / "independent_gate_review.md"
PRETRAINED_CONFIG_PATH = ROOT / "configs" / "foundation_stage2_pretrained_v1.json"
PRETRAINED_CHECKPOINT_PATH = ROOT / "models" / "pretrained" / STAGE2_SCHEMA / "convnextv2_tiny_1k_224_fcmae.pt"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def canonical_sha256(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def tensor_state_sha256(state: dict) -> str:
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(value.dtype).encode("ascii"))
        h.update(np.asarray(value.shape, dtype=np.int64).tobytes())
        h.update(value.numpy().tobytes())
    return h.hexdigest()


def stable_seed(*parts) -> int:
    token = "|".join(str(p) for p in parts).encode("utf-8")
    return int.from_bytes(hashlib.sha256(token).digest()[:8], "little") % (2**32)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def augment_drr(array: np.ndarray, sample_id: str, view: str, fold: int, epoch: int, worker_id: int) -> np.ndarray:
    """Deterministic training-only photometric augmentation; geometry is never changed."""
    rng = np.random.default_rng(stable_seed(SEED, fold, epoch, worker_id, sample_id, view))
    gamma = rng.uniform(*AUGMENTATION["gamma"])
    brightness = rng.uniform(*AUGMENTATION["brightness"])
    sigma = rng.uniform(*AUGMENTATION["gaussian_noise_sigma"])
    out = np.power(np.clip(array, 0.0, 1.0), gamma, dtype=np.float32)
    out = out + np.float32(brightness)
    if sigma > 0:
        out = out + rng.normal(0.0, sigma, size=out.shape).astype(np.float32)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def require_stage1_pass() -> str:
    if not STAGE1_GATE_PATH.is_file():
        raise FileNotFoundError(f"Stage 1 Agent-N gate record is missing: {STAGE1_GATE_PATH}")
    review = STAGE1_GATE_PATH.read_text(encoding="utf-8")
    if "- Reviewer: Agent N" not in review or "- Verdict: PASS" not in review:
        raise RuntimeError("Stage 1 independent gate does not contain an Agent-N PASS")
    return sha256_file(STAGE1_GATE_PATH)


def verify_pretrained_checkpoint() -> dict:
    if not PRETRAINED_CONFIG_PATH.is_file():
        raise FileNotFoundError(f"pinned pretrained configuration is missing: {PRETRAINED_CONFIG_PATH}")
    config = json.loads(PRETRAINED_CONFIG_PATH.read_text(encoding="utf-8"))
    if config.get("model_tag") != PRETRAIN_MODEL:
        raise RuntimeError("pinned pretrained model tag mismatch")
    if not PRETRAINED_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"stage the pinned FCMAE checkpoint before execution: {PRETRAINED_CHECKPOINT_PATH}")
    expected_sha = config.get("sha256")
    expected_size = config.get("size_bytes")
    if not expected_sha or expected_size in (None, 0):
        raise RuntimeError("record the staged pretrained file SHA-256 and size in foundation_stage2_pretrained_v1.json")
    actual_size = PRETRAINED_CHECKPOINT_PATH.stat().st_size
    actual_sha = sha256_file(PRETRAINED_CHECKPOINT_PATH)
    if actual_size != int(expected_size) or actual_sha != expected_sha:
        raise RuntimeError("pinned pretrained checkpoint size/SHA-256 mismatch")
    return {**config, "path": str(PRETRAINED_CHECKPOINT_PATH), "verified_sha256": actual_sha}


def load_certified_folds(fold: int):
    stage1_gate_sha = require_stage1_pass()
    baseline = json.loads(BASELINE_CONFIG_PATH.read_text(encoding="utf-8"))
    data_contract = json.loads(DATA_CONFIG_PATH.read_text(encoding="utf-8"))
    meta = json.loads(MANIFEST_META_PATH.read_text(encoding="utf-8"))
    if baseline["protocol_version"] != PROTOCOL_VERSION:
        raise RuntimeError("baseline protocol version mismatch")
    if not meta.get("certification_approved", False):
        raise RuntimeError("Stage 1 is not certified: certification_approved is false")
    if int(meta.get("ready_rows", -1)) != 71 or int(meta.get("pending_recertification_rows", -1)) != 0:
        raise RuntimeError("Stage 1 must contain exactly 71 ready rows and zero pending rows")
    leakage = meta.get("leakage", {})
    leakage_fields = ("subject_multiple_test_folds", ("fold" + str(5) + "_rows"), "augmentation_parent_mismatch")
    if any(int(leakage.get(field, -1)) != 0 for field in leakage_fields):
        raise RuntimeError(f"certified leakage report is not zero: {leakage}")
    if int(leakage.get("derived_split_subject_overlap", -1)) != 0:
        raise RuntimeError("certified derived split overlap is not zero")
    actual_manifest_sha = sha256_file(MANIFEST_PATH)
    if actual_manifest_sha != meta.get("sha256"):
        raise RuntimeError("manifest SHA-256 does not match quantitative_manifest_v1.metadata.json")

    all_rows = pd.read_csv(MANIFEST_PATH, dtype={"test_fold": "Int64"})
    ready = all_rows[all_rows["status"].eq("ready")].copy()
    if len(ready) != 71:
        raise RuntimeError(f"expected 71 ready manifest rows, found {len(ready)}")
    counts = ready.groupby("dataset").size().to_dict()
    if counts != {"Ruikar": 13, "VSD": 58}:
        raise RuntimeError(f"cohort mismatch: {counts}")
    if set(ready["test_fold"].astype(int)) != set(range(5)):
        raise RuntimeError("test_fold must contain only 0..4")
    if ready.groupby("subject_id")["test_fold"].nunique().max() != 1:
        raise RuntimeError("subject leakage across test folds")
    parent_rows = ready.set_index("sample_id")[["subject_id", "test_fold"]]
    for row in ready.itertuples(index=False):
        parent_id = row.augmentation_parent
        if parent_id not in parent_rows.index:
            raise RuntimeError(f"augmentation parent is absent from certified rows: {row.sample_id} -> {parent_id}")
        parent = parent_rows.loc[parent_id]
        if parent["subject_id"] != row.subject_id or int(parent["test_fold"]) != int(row.test_fold):
            raise RuntimeError(f"augmentation leakage: {row.sample_id} -> {parent_id}")
    if ready["orientation"].ne("LPS").any():
        raise RuntimeError("non-LPS row in certified manifest")
    if ready["target_version"].ne(data_contract["target_version"]).any():
        raise RuntimeError("target version mismatch")
    if ready["drr_version"].ne(data_contract["drr_version"]).any():
        raise RuntimeError("DRR version mismatch")

    ready["test_fold"] = ready["test_fold"].astype(int)
    ready["split"] = "train"
    ready.loc[ready["test_fold"].eq(fold), "split"] = "test"
    ready.loc[ready["test_fold"].eq((fold + 1) % 5), "split"] = "validation"
    split_subjects = {name: set(group["subject_id"]) for name, group in ready.groupby("split")}
    if split_subjects["train"] & split_subjects["validation"] or split_subjects["train"] & split_subjects["test"] or split_subjects["validation"] & split_subjects["test"]:
        raise RuntimeError("subject overlap between train/validation/test")

    # Only train and validation paths are checked/opened in Stage 2. Test paths remain unopened.
    for split in ("train", "validation"):
        for row in ready[ready["split"].eq(split)].itertuples(index=False):
            for field in ("ap_drr_path", "lat_drr_path"):
                path = ROOT / getattr(row, field)
                if not path.is_file():
                    raise FileNotFoundError(f"missing certified {split} input: {path}")

    return ready, ready[ready.split.eq("train")].copy(), ready[ready.split.eq("validation")].copy(), ready[ready.split.eq("test")].copy(), meta


def read_clean_drr(path: Path) -> np.ndarray:
    array = np.load(path).astype(np.float32)
    if array.shape != (IMAGE_SIZE, IMAGE_SIZE):
        raise ValueError(f"DRR shape must be 256x256, got {array.shape} for {path}")
    if not np.isfinite(array).all() or array.min() < -1e-6 or array.max() > 1.0 + 1e-6:
        raise ValueError(f"DRR must be finite and within [0,1]: {path}")
    return np.clip(array, 0.0, 1.0)


class AnchorDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, training: bool):
        self.training = training
        self.epoch = 0
        self.records = []
        for row in rows.itertuples(index=False):
            self.records.extend([
                (row.sample_id, "ap", ROOT / row.ap_drr_path),
                (row.sample_id, "lat", ROOT / row.lat_drr_path),
            ])

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        sample_id, view, path = self.records[index]
        array = read_clean_drr(path)
        worker = get_worker_info()
        worker_id = 0 if worker is None else worker.id
        if self.training:
            array = augment_drr(array, sample_id, view, FOLD, self.epoch, worker_id)
        return {"image": torch.from_numpy(array).unsqueeze(0), "sample_id": sample_id, "view": view}


class PairDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, training: bool):
        self.rows = rows.reset_index(drop=True)
        self.training = training
        self.epoch = 0

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        worker = get_worker_info()
        worker_id = 0 if worker is None else worker.id
        ap = read_clean_drr(ROOT / row.ap_drr_path)
        lat = read_clean_drr(ROOT / row.lat_drr_path)
        if self.training:
            ap = augment_drr(ap, row.sample_id, "ap", FOLD, self.epoch, worker_id)
            lat = augment_drr(lat, row.sample_id, "lat", FOLD, self.epoch, worker_id)
        return {"ap": torch.from_numpy(ap).unsqueeze(0), "lat": torch.from_numpy(lat).unsqueeze(0), "sample_id": row.sample_id}


class EpochUniformReplacementSampler(Sampler):
    """Exactly EFFECTIVE_BATCH uniformly sampled training records per epoch."""
    def __init__(self, data_source, stage: str, num_samples: int = EFFECTIVE_BATCH):
        if len(data_source) < 1: raise ValueError("replacement sampler requires a non-empty dataset")
        self.data_source = data_source
        self.stage = str(stage)
        self.num_samples = int(num_samples)
        self.epoch = 0

    def set_epoch(self, epoch: int): self.epoch = int(epoch)
    def __len__(self): return self.num_samples

    def __iter__(self):
        generator = make_generator(stable_seed(SEED, FOLD, self.stage, self.epoch, "uniform_replacement"))
        indices = torch.randint(len(self.data_source), (self.num_samples,), generator=generator)
        return iter(indices.tolist())


seed_everything()
print("project root:", ROOT)

In [ ]:
MASK_GRID = IMAGE_SIZE // PATCH_SIZE


def build_backbone(pretrained: bool):
    """No fallback is permitted: the locally pinned FCMAE file must load or the run fails."""
    kwargs = {"features_only": True}
    if pretrained:
        verify_pretrained_checkpoint()
        kwargs["pretrained_cfg_overlay"] = {"file": str(PRETRAINED_CHECKPOINT_PATH)}
    return timm.create_model(PRETRAIN_MODEL, pretrained=pretrained, **kwargs)


class FCMAEEncoder(nn.Module):
    """Dense-mask equivalent of the sparse FCMAE encoder, with re-zeroing after every block."""
    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone
        required = ["stem_0", "stem_1", *[f"stages_{i}" for i in range(4)]]
        missing = [name for name in required if not hasattr(backbone, name)]
        if missing:
            raise RuntimeError(f"unsupported timm features_only wrapper; missing {missing}")

    @staticmethod
    def _visible(x, mask):
        visible = F.interpolate((~mask).float(), size=x.shape[-2:], mode="nearest")
        return x * visible

    def extract_pyramid(self, x):
        """Return clean, unmasked L0-L3 feature maps for representation audits."""
        bb = self.backbone
        x = bb.stem_1(bb.stem_0(x))
        features = []
        for index in range(4):
            stage = getattr(bb, f"stages_{index}")
            x = stage.downsample(x)
            for block in stage.blocks: x = block(x)
            features.append(x)
        return features

    def forward(self, x, mask):
        bb = self.backbone
        x = self._visible(x, mask)
        x = bb.stem_1(bb.stem_0(x))
        x = self._visible(x, mask)
        features = []
        for index in range(4):
            stage = getattr(bb, f"stages_{index}")
            x = stage.downsample(x)
            x = self._visible(x, mask)
            for block in stage.blocks:
                x = block(x)
                x = self._visible(x, mask)
            features.append(x)
        return features


class ConvNeXtDecoderBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.depthwise = nn.Conv2d(channels, channels, 7, padding=3, groups=channels)
        self.norm = nn.GroupNorm(1, channels)
        self.pointwise1 = nn.Conv2d(channels, 4 * channels, 1)
        self.pointwise2 = nn.Conv2d(4 * channels, channels, 1)

    def forward(self, x):
        residual = x
        x = self.depthwise(x)
        x = self.norm(x)
        x = self.pointwise2(F.gelu(self.pointwise1(x)))
        return x + residual


class FCMAEDecoder(nn.Module):
    def __init__(self, input_dim=768, decoder_dim=DECODER_DIM):
        super().__init__()
        self.projection = nn.Conv2d(input_dim, decoder_dim, 1)
        self.block = ConvNeXtDecoderBlock(decoder_dim)
        self.prediction = nn.Conv2d(decoder_dim, 3 * PATCH_SIZE * PATCH_SIZE, 1)

    def forward(self, feature):
        pixels = self.prediction(self.block(self.projection(feature)))
        batch = pixels.shape[0]
        pixels = pixels.view(batch, 3, PATCH_SIZE, PATCH_SIZE, MASK_GRID, MASK_GRID)
        return pixels.permute(0, 1, 4, 5, 2, 3).contiguous()


class CrossViewBlock(nn.Module):
    def __init__(self, dim=768, heads=8):
        super().__init__()
        self.self_norm = nn.LayerNorm(dim)
        self.self_attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.cross_norm = nn.LayerNorm(dim)
        self.cross_attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp_norm = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, target, source, attention_mask):
        query = self.self_norm(target)
        target = target + self.self_attention(query, query, query, need_weights=False)[0]
        query = self.cross_norm(target)
        target = target + self.cross_attention(query, source, source, attn_mask=attention_mask, need_weights=False)[0]
        return target + self.mlp(self.mlp_norm(target))


class CrossViewDecoder(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.block = CrossViewBlock(dim)
        self.prediction = nn.Linear(dim, 3 * PATCH_SIZE * PATCH_SIZE)

    def forward(self, target_feature, source_feature, attention_mask):
        batch, channels, height, width = target_feature.shape
        target = target_feature.flatten(2).transpose(1, 2)
        source = source_feature.flatten(2).transpose(1, 2)
        pixels = self.prediction(self.block(target, source, attention_mask))
        pixels = pixels.view(batch, height, width, 3, PATCH_SIZE, PATCH_SIZE)
        return pixels.permute(0, 3, 1, 2, 4, 5).contiguous()


def rowwise_attention_mask(device):
    index = torch.arange(MASK_GRID * MASK_GRID, device=device)
    rows = index // MASK_GRID
    mask = torch.zeros((index.numel(), index.numel()), device=device)
    mask[(rows[:, None] - rows[None, :]).abs() > XVIEW_ROW_SLACK] = float("-inf")
    return mask


def uniform_mask(batch_size: int, device, generator: torch.Generator):
    count = MASK_GRID * MASK_GRID
    masked = int(round(MASK_RATIO * count))
    noise = torch.rand((batch_size, count), generator=generator, device="cpu")
    selected = noise.argsort(dim=1)[:, :masked]
    mask = torch.zeros((batch_size, count), dtype=torch.bool)
    mask.scatter_(1, selected, True)
    return mask.view(batch_size, 1, MASK_GRID, MASK_GRID).to(device)


def patch_normalized_target(raw_one_channel):
    raw_three_channel = raw_one_channel.repeat(1, 3, 1, 1)
    patches = raw_three_channel.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
    mean = patches.mean(dim=(-1, -2), keepdim=True)
    variance = patches.var(dim=(-1, -2), keepdim=True, unbiased=False)
    return (patches - mean) / torch.sqrt(variance + 1e-6)


def masked_mse(prediction, target, mask):
    weights = mask.unsqueeze(-1).unsqueeze(-1).float()
    return (((prediction - target) ** 2) * weights).sum() / (weights.sum() * prediction.shape[1] * PATCH_SIZE * PATCH_SIZE).clamp_min(1.0)


def encoder_input(raw, mean, std):
    image = raw.repeat(1, 3, 1, 1)
    mean_tensor = torch.as_tensor(mean, dtype=image.dtype, device=image.device).view(1, 3, 1, 1)
    std_tensor = torch.as_tensor(std, dtype=image.dtype, device=image.device).view(1, 3, 1, 1)
    return (image - mean_tensor) / std_tensor


def p1_forward(encoder, decoder, raw, mean, std, generator):
    raw = raw.to(DEVICE, non_blocking=True)
    mask = uniform_mask(raw.shape[0], DEVICE, generator)
    feature = encoder(encoder_input(raw, mean, std), mask)[-1]
    prediction = decoder(feature)
    loss = masked_mse(prediction, patch_normalized_target(raw), mask)
    return loss, feature


def p2_forward(encoder, fcmae_decoder, cross_decoder, ap, lat, mean, std, generator):
    ap = ap.to(DEVICE, non_blocking=True)
    lat = lat.to(DEVICE, non_blocking=True)
    ap_mask = uniform_mask(ap.shape[0], DEVICE, generator)
    lat_mask = uniform_mask(lat.shape[0], DEVICE, generator)
    ap_feature = encoder(encoder_input(ap, mean, std), ap_mask)[-1]
    lat_feature = encoder(encoder_input(lat, mean, std), lat_mask)[-1]
    ap_target = patch_normalized_target(ap)
    lat_target = patch_normalized_target(lat)
    fcmae = 0.5 * (
        masked_mse(fcmae_decoder(ap_feature), ap_target, ap_mask)
        + masked_mse(fcmae_decoder(lat_feature), lat_target, lat_mask)
    )
    attention_mask = rowwise_attention_mask(DEVICE)
    ap_from_lat = masked_mse(cross_decoder(ap_feature, lat_feature, attention_mask), ap_target, ap_mask)
    lat_from_ap = masked_mse(cross_decoder(lat_feature, ap_feature, attention_mask), lat_target, lat_mask)
    cross = 0.5 * (ap_from_lat + lat_from_ap)
    return fcmae + XVIEW_WEIGHT * cross, {"fcmae": fcmae.detach(), "cross": cross.detach()}, (ap_feature, lat_feature)

In [ ]:
def make_generator(seed: int) -> torch.Generator:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return generator


def amp_context():
    if not (USE_AMP and DEVICE.type == "cuda"):
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type="cuda", dtype=dtype)


def make_scaler():
    enabled = USE_AMP and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported()
    return torch.amp.GradScaler("cuda", enabled=enabled)


def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def scheduler_for(optimizer, total_updates: int):
    warmup = max(1, int(round(WARMUP_FRACTION * total_updates)))
    def multiplier(step):
        if step < warmup:
            return max(1e-8, (step + 1) / warmup)
        progress = (step - warmup) / max(1, total_updates - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)


def save_training_state(path, stage, epoch, global_step, modules, optimizer, scheduler, scaler, best_metric, config_sha):
    payload = {
        "schema_version": STAGE2_SCHEMA,
        "stage": stage,
        "fold": FOLD,
        "epoch": epoch,
        "global_step": global_step,
        "modules": {name: module.state_dict() for name, module in modules.items()},
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "rng_state": rng_state(),
        "best_metric": float(best_metric),
        "config_sha256": config_sha,
    }
    torch.save(payload, path)


def strict_load(module, state, label):
    incompatible = module.load_state_dict(state, strict=True)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError(f"{label} strict-load failure: {incompatible}")


def resource_usage(started):
    peak_host = None
    if resource is not None:
        maximum_rss = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        peak_host = maximum_rss if platform.system() == "Darwin" else maximum_rss * 1024
    peak_gpu = int(torch.cuda.max_memory_allocated()) if DEVICE.type == "cuda" else None
    total_gpu = int(torch.cuda.get_device_properties(DEVICE).total_memory) if DEVICE.type == "cuda" else None
    return {"wall_seconds": round(time.time() - started, 1), "peak_gpu_bytes": peak_gpu, "total_gpu_bytes": total_gpu, "peak_host_bytes": peak_host}


def save_training_curve(history, path, stage):
    frame = pd.DataFrame(history)
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.plot(frame["epoch"], frame["train_loss"], label="train")
    axis.plot(frame["epoch"], frame["validation_loss"], label="validation")
    axis.set(xlabel="epoch", ylabel="loss", title=f"{stage} fold {FOLD}")
    axis.grid(alpha=0.25); axis.legend(); figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def run_config(stage, manifest_meta, train_rows, validation_rows, test_rows, upstream_sha=None):
    return {
        "schema_version": STAGE2_SCHEMA,
        "protocol_version": PROTOCOL_VERSION,
        "stage": stage,
        "fold": FOLD,
        "seed": SEED,
        "manifest_sha256": manifest_meta["sha256"],
        "baseline_protocol_sha256": sha256_file(BASELINE_CONFIG_PATH),
        "data_contract_sha256": sha256_file(DATA_CONFIG_PATH),
        "stage1_agent_n_gate_sha256": sha256_file(STAGE1_GATE_PATH),
        "pretrained_source": json.loads(PRETRAINED_CONFIG_PATH.read_text(encoding="utf-8")),
        "train_sample_ids": sorted(train_rows.sample_id.tolist()),
        "validation_sample_ids": sorted(validation_rows.sample_id.tolist()),
        "test_sample_ids_not_opened": sorted(test_rows.sample_id.tolist()),
        "pretrained_model": PRETRAIN_MODEL,
        "upstream_checkpoint_sha256": upstream_sha,
        "augmentation": AUGMENTATION,
        "hyperparameters": {
            "mask_ratio": MASK_RATIO,
            "patch_size": PATCH_SIZE,
            "decoder_dim": DECODER_DIM,
            "effective_batch": EFFECTIVE_BATCH,
            "micro_batch": MICRO_BATCH,
            "accum_steps": ACCUM_STEPS,
            "training_sampler": "deterministic_uniform_with_replacement_exactly_128_draws_per_epoch",
            "peak_lr": PEAK_LR,
            "weight_decay": WEIGHT_DECAY,
            "warmup_fraction": WARMUP_FRACTION,
            "p1_max_epochs": P1_MAX_EPOCHS,
            "p2_max_epochs": P2_MAX_EPOCHS,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "xview_weight": XVIEW_WEIGHT,
            "xview_row_slack": XVIEW_ROW_SLACK,
        },
        "software": {"python": platform.python_version(), "torch": torch.__version__, "timm": timm.__version__},
        "hardware": {"device": str(DEVICE), "cuda_device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None},
        "output_paths": {"artifact_root": str(ARTIFACT_ROOT)},
        "oom_policy": OOM_POLICY,
    }


def write_json(path: Path, payload: dict):
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def train_p1(train_rows, validation_rows, test_rows, manifest_meta):
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True); started = time.time()
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    config = run_config("FCMAE_P1", manifest_meta, train_rows, validation_rows, test_rows)
    backbone = build_backbone(pretrained=True)
    pretrained_cfg = dict(backbone.pretrained_cfg)
    mean, std = pretrained_cfg["mean"], pretrained_cfg["std"]
    initialization_sha = tensor_state_sha256(backbone.state_dict())
    config["pretrained_configuration"] = pretrained_cfg
    config["pretrained_state_sha256"] = initialization_sha
    config_sha = canonical_sha256(config)
    write_json(ARTIFACT_ROOT / "fcmae_p1_config.json", config)

    encoder = FCMAEEncoder(backbone).to(DEVICE)
    decoder = FCMAEDecoder().to(DEVICE)
    modules = {"encoder": encoder, "fcmae_decoder": decoder}
    optimizer = torch.optim.AdamW(
        [parameter for module in modules.values() for parameter in module.parameters()],
        lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY,
    )
    train_set = AnchorDataset(train_rows, training=True)
    validation_set = AnchorDataset(validation_rows, training=False)
    train_sampler = EpochUniformReplacementSampler(train_set, "p1")
    train_loader = DataLoader(train_set, batch_size=MICRO_BATCH, sampler=train_sampler, num_workers=NUM_WORKERS, drop_last=False)
    validation_loader = DataLoader(validation_set, batch_size=MICRO_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    if len(train_sampler) != EFFECTIVE_BATCH or len(train_loader) != ACCUM_STEPS:
        raise RuntimeError("P1 loader must yield exactly eight micro-batches and 128 draws")
    updates_per_epoch = 1
    scheduler = scheduler_for(optimizer, P1_MAX_EPOCHS * updates_per_epoch)
    scaler = make_scaler()
    start_epoch, global_step, best, patience, history = 0, 0, float("inf"), 0, []

    if RESUME_P1:
        checkpoint = torch.load(RESUME_P1, map_location=DEVICE, weights_only=False)
        if checkpoint["config_sha256"] != config_sha or checkpoint["fold"] != FOLD:
            raise RuntimeError("P1 resume checkpoint config/fold mismatch")
        for name, module in modules.items(): strict_load(module, checkpoint["modules"][name], name)
        optimizer.load_state_dict(checkpoint["optimizer"]); scheduler.load_state_dict(checkpoint["scheduler"]); scaler.load_state_dict(checkpoint["scaler"])
        restore_rng_state(checkpoint["rng_state"])
        start_epoch, global_step, best = checkpoint["epoch"] + 1, checkpoint["global_step"], checkpoint["best_metric"]

    initial_validation = None
    for epoch in range(start_epoch, P1_MAX_EPOCHS):
        train_set.set_epoch(epoch); train_sampler.set_epoch(epoch)
        encoder.train(); decoder.train(); optimizer.zero_grad(set_to_none=True)
        train_total, train_count = 0.0, 0
        for index, batch in enumerate(train_loader):
            generator = make_generator(stable_seed(SEED, FOLD, "p1", epoch, index))
            with amp_context(): loss, _ = p1_forward(encoder, decoder, batch["image"], mean, std, generator)
            if not torch.isfinite(loss): raise FloatingPointError("non-finite P1 loss")
            scaler.scale(loss / ACCUM_STEPS).backward()
            if (index + 1) % ACCUM_STEPS == 0 or index + 1 == len(train_loader):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_([p for m in modules.values() for p in m.parameters()], 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step(); global_step += 1
            train_total += loss.item() * batch["image"].shape[0]; train_count += batch["image"].shape[0]

        encoder.eval(); decoder.eval(); validation_total, validation_count, feature_std = 0.0, 0, []
        with torch.no_grad():
            for index, batch in enumerate(validation_loader):
                generator = make_generator(stable_seed(SEED, FOLD, "p1_validation", index))
                with amp_context(): loss, feature = p1_forward(encoder, decoder, batch["image"], mean, std, generator)
                validation_total += loss.item() * batch["image"].shape[0]; validation_count += batch["image"].shape[0]
                feature_std.append(float(feature.float().std().item()))
        validation_loss = validation_total / validation_count
        mean_feature_std = float(np.mean(feature_std))
        if not np.isfinite(mean_feature_std) or mean_feature_std <= FEATURE_STD_MIN:
            raise RuntimeError(f"collapsed P1 validation features: std={mean_feature_std}")
        if initial_validation is None: initial_validation = validation_loss
        record = {"epoch": epoch, "train_loss": train_total / train_count, "validation_loss": validation_loss, "lr": optimizer.param_groups[0]["lr"], "feature_std": mean_feature_std}
        history.append(record); pd.DataFrame(history).to_csv(ARTIFACT_ROOT / "fcmae_p1_history.csv", index=False)
        if validation_loss < best:
            best, patience = validation_loss, 0
            save_training_state(ARTIFACT_ROOT / "fcmae_p1_best_trainstate.pth", "FCMAE_P1", epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        else:
            patience += 1
        save_training_state(ARTIFACT_ROOT / "fcmae_p1_last_trainstate.pth", "FCMAE_P1", epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            save_training_state(ARTIFACT_ROOT / f"fcmae_p1_epoch{epoch + 1:03d}_trainstate.pth", "FCMAE_P1", epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        print(record)
        if patience >= EARLY_STOP_PATIENCE: break

    best_checkpoint = torch.load(ARTIFACT_ROOT / "fcmae_p1_best_trainstate.pth", map_location="cpu", weights_only=False)
    strict_load(encoder.cpu(), best_checkpoint["modules"]["encoder"], "P1 encoder")
    if not best < initial_validation:
        raise RuntimeError(f"P1 did not improve over epoch-zero validation: initial={initial_validation}, best={best}")
    export = {"schema_version": STAGE2_SCHEMA, "stage": "FCMAE_P1", "fold": FOLD, "encoder_state": encoder.backbone.state_dict(), "config_sha256": config_sha, "manifest_sha256": manifest_meta["sha256"], "pretrained_state_sha256": initialization_sha}
    torch.save(export, ARTIFACT_ROOT / "fcmae_p1_encoder.pth")
    save_training_curve(history, ARTIFACT_ROOT / "fcmae_p1_training_curve.png", "FCMAE P1")
    provenance = {**config, "config_sha256": config_sha, "best_validation_loss": best, "initial_validation_loss": initial_validation, "encoder_sha256": tensor_state_sha256(export["encoder_state"]), "trainstate_sha256": sha256_file(ARTIFACT_ROOT / "fcmae_p1_best_trainstate.pth"), "resource_usage": resource_usage(started), "qa_figures": [str(ARTIFACT_ROOT / "fcmae_p1_training_curve.png")]}
    write_json(ARTIFACT_ROOT / "fcmae_p1_provenance.json", provenance)
    return provenance


def crossview_pairing_probe(encoder, cross_decoder, validation_rows, mean, std):
    loader = DataLoader(PairDataset(validation_rows, training=False), batch_size=min(8, len(validation_rows)), shuffle=False)
    batch = next(iter(loader))
    if batch["ap"].shape[0] < 2: raise RuntimeError("pairing probe requires at least two validation pairs")
    encoder.eval(); cross_decoder.eval()
    generator = make_generator(stable_seed(SEED, FOLD, "pair_probe"))
    ap = batch["ap"].to(DEVICE); lat = batch["lat"].to(DEVICE)
    ap_mask = uniform_mask(ap.shape[0], DEVICE, generator); lat_mask = uniform_mask(lat.shape[0], DEVICE, generator)
    with torch.no_grad(), amp_context():
        ap_feature = encoder(encoder_input(ap, mean, std), ap_mask)[-1]
        lat_feature = encoder(encoder_input(lat, mean, std), lat_mask)[-1]
        ap_target = patch_normalized_target(ap); lat_target = patch_normalized_target(lat); attention = rowwise_attention_mask(DEVICE)
        paired = 0.5 * (
            masked_mse(cross_decoder(ap_feature, lat_feature, attention), ap_target, ap_mask)
            + masked_mse(cross_decoder(lat_feature, ap_feature, attention), lat_target, lat_mask)
        )
        shuffled = 0.5 * (
            masked_mse(cross_decoder(ap_feature, lat_feature.roll(1, 0), attention), ap_target, ap_mask)
            + masked_mse(cross_decoder(lat_feature, ap_feature.roll(1, 0), attention), lat_target, lat_mask)
        )
    return float(paired), float(shuffled)


def train_p2(train_rows, validation_rows, test_rows, manifest_meta):
    started = time.time()
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    p1_export_path = ARTIFACT_ROOT / "fcmae_p1_encoder.pth"
    p1_state_path = ARTIFACT_ROOT / "fcmae_p1_best_trainstate.pth"
    if not p1_export_path.is_file() or not p1_state_path.is_file():
        raise FileNotFoundError("P2 requires this fold's approved P1 export and best trainstate")
    p1_export = torch.load(p1_export_path, map_location="cpu", weights_only=False)
    if p1_export["fold"] != FOLD or p1_export["manifest_sha256"] != manifest_meta["sha256"]:
        raise RuntimeError("P1 export fold/manifest mismatch")
    config = run_config("cross_view_P2", manifest_meta, train_rows, validation_rows, test_rows, sha256_file(p1_export_path))
    config_sha = canonical_sha256(config); write_json(ARTIFACT_ROOT / "fcmae_p2_config.json", config)

    backbone = build_backbone(pretrained=False)
    strict_load(backbone, p1_export["encoder_state"], "P1 backbone -> P2")
    mean, std = backbone.pretrained_cfg["mean"], backbone.pretrained_cfg["std"]
    encoder = FCMAEEncoder(backbone).to(DEVICE)
    fcmae_decoder = FCMAEDecoder().to(DEVICE)
    p1_trainstate = torch.load(p1_state_path, map_location="cpu", weights_only=False)
    strict_load(fcmae_decoder, p1_trainstate["modules"]["fcmae_decoder"], "P1 decoder -> P2")
    cross_decoder = CrossViewDecoder().to(DEVICE)
    modules = {"encoder": encoder, "fcmae_decoder": fcmae_decoder, "cross_decoder": cross_decoder}
    optimizer = torch.optim.AdamW([p for m in modules.values() for p in m.parameters()], lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
    train_set = PairDataset(train_rows, training=True); validation_set = PairDataset(validation_rows, training=False)
    train_sampler = EpochUniformReplacementSampler(train_set, "p2")
    train_loader = DataLoader(train_set, batch_size=MICRO_BATCH, sampler=train_sampler, num_workers=NUM_WORKERS)
    validation_loader = DataLoader(validation_set, batch_size=MICRO_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    if len(train_sampler) != EFFECTIVE_BATCH or len(train_loader) != ACCUM_STEPS:
        raise RuntimeError("P2 loader must yield exactly eight micro-batches and 128 draws")
    updates_per_epoch = 1
    scheduler = scheduler_for(optimizer, P2_MAX_EPOCHS * updates_per_epoch); scaler = make_scaler()
    start_epoch, global_step, best, patience, initial_validation, history = 0, 0, float("inf"), 0, None, []

    if RESUME_P2:
        checkpoint = torch.load(RESUME_P2, map_location=DEVICE, weights_only=False)
        if checkpoint["config_sha256"] != config_sha or checkpoint["fold"] != FOLD: raise RuntimeError("P2 resume mismatch")
        for name, module in modules.items(): strict_load(module, checkpoint["modules"][name], name)
        optimizer.load_state_dict(checkpoint["optimizer"]); scheduler.load_state_dict(checkpoint["scheduler"]); scaler.load_state_dict(checkpoint["scaler"]); restore_rng_state(checkpoint["rng_state"])
        start_epoch, global_step, best = checkpoint["epoch"] + 1, checkpoint["global_step"], checkpoint["best_metric"]

    for epoch in range(start_epoch, P2_MAX_EPOCHS):
        train_set.set_epoch(epoch)
        train_sampler.set_epoch(epoch)
        for module in modules.values(): module.train()
        optimizer.zero_grad(set_to_none=True); totals = {"loss": 0.0, "fcmae": 0.0, "cross": 0.0, "count": 0}
        for index, batch in enumerate(train_loader):
            generator = make_generator(stable_seed(SEED, FOLD, "p2", epoch, index))
            with amp_context(): loss, pieces, _ = p2_forward(encoder, fcmae_decoder, cross_decoder, batch["ap"], batch["lat"], mean, std, generator)
            if not torch.isfinite(loss): raise FloatingPointError("non-finite P2 loss")
            scaler.scale(loss / ACCUM_STEPS).backward()
            if (index + 1) % ACCUM_STEPS == 0 or index + 1 == len(train_loader):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_([p for m in modules.values() for p in m.parameters()], 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step(); global_step += 1
            n = batch["ap"].shape[0]; totals["loss"] += loss.item() * n; totals["fcmae"] += float(pieces["fcmae"]) * n; totals["cross"] += float(pieces["cross"]) * n; totals["count"] += n

        for module in modules.values(): module.eval()
        validation_total, validation_count, feature_std = 0.0, 0, []
        with torch.no_grad():
            for index, batch in enumerate(validation_loader):
                generator = make_generator(stable_seed(SEED, FOLD, "p2_validation", index))
                with amp_context(): loss, _, features = p2_forward(encoder, fcmae_decoder, cross_decoder, batch["ap"], batch["lat"], mean, std, generator)
                n = batch["ap"].shape[0]; validation_total += loss.item() * n; validation_count += n; feature_std.extend([float(f.float().std()) for f in features])
        validation_loss = validation_total / validation_count
        mean_feature_std = float(np.mean(feature_std))
        if not np.isfinite(mean_feature_std) or mean_feature_std <= FEATURE_STD_MIN:
            raise RuntimeError(f"collapsed P2 validation features: std={mean_feature_std}")
        if initial_validation is None: initial_validation = validation_loss
        record = {"epoch": epoch, "train_loss": totals["loss"] / totals["count"], "train_fcmae": totals["fcmae"] / totals["count"], "train_cross": totals["cross"] / totals["count"], "validation_loss": validation_loss, "feature_std": mean_feature_std, "lr": optimizer.param_groups[0]["lr"]}
        history.append(record); pd.DataFrame(history).to_csv(ARTIFACT_ROOT / "fcmae_p2_history.csv", index=False)
        if validation_loss < best:
            best, patience = validation_loss, 0
            save_training_state(ARTIFACT_ROOT / "fcmae_p2_best_trainstate.pth", "cross_view_P2", epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        else: patience += 1
        save_training_state(ARTIFACT_ROOT / "fcmae_p2_last_trainstate.pth", "cross_view_P2", epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        if (epoch + 1) % CHECKPOINT_EVERY == 0: save_training_state(ARTIFACT_ROOT / f"fcmae_p2_epoch{epoch + 1:03d}_trainstate.pth", "cross_view_P2", epoch, global_step, modules, optimizer, scheduler, scaler, best, config_sha)
        print(record)
        if patience >= EARLY_STOP_PATIENCE: break

    best_checkpoint = torch.load(ARTIFACT_ROOT / "fcmae_p2_best_trainstate.pth", map_location="cpu", weights_only=False)
    strict_load(encoder.cpu(), best_checkpoint["modules"]["encoder"], "P2 encoder")
    strict_load(cross_decoder.cpu(), best_checkpoint["modules"]["cross_decoder"], "P2 cross decoder")
    encoder.to(DEVICE); cross_decoder.to(DEVICE)
    paired, shuffled = crossview_pairing_probe(encoder, cross_decoder, validation_rows, mean, std)
    if not best < initial_validation: raise RuntimeError("P2 validation loss did not improve")
    if not paired < shuffled: raise RuntimeError(f"pairing gate failed: paired={paired}, shuffled={shuffled}")
    export_state = {key: value.cpu() for key, value in encoder.backbone.state_dict().items()}
    export = {"schema_version": STAGE2_SCHEMA, "stage": "cross_view_P2", "fold": FOLD, "encoder_state": export_state, "config_sha256": config_sha, "manifest_sha256": manifest_meta["sha256"], "p1_encoder_sha256": sha256_file(p1_export_path)}
    torch.save(export, ARTIFACT_ROOT / "fcmae_p2_encoder.pth")
    save_training_curve(history, ARTIFACT_ROOT / "fcmae_p2_training_curve.png", "Cross-view P2")
    provenance = {**config, "config_sha256": config_sha, "initial_validation_loss": initial_validation, "best_validation_loss": best, "paired_crossview_loss": paired, "shuffled_crossview_loss": shuffled, "encoder_sha256": tensor_state_sha256(export_state), "trainstate_sha256": sha256_file(ARTIFACT_ROOT / "fcmae_p2_best_trainstate.pth"), "resource_usage": resource_usage(started), "qa_figures": [str(ARTIFACT_ROOT / "fcmae_p2_training_curve.png")]}
    write_json(ARTIFACT_ROOT / "fcmae_p2_provenance.json", provenance)
    return provenance


def run_preflight(train_rows, validation_rows, test_rows, manifest_meta):
    """One exact update plus clean validation and strict reload; learned weights are discarded."""
    started = time.time()
    preflight_root = ARTIFACT_ROOT / "preflight"
    preflight_root.mkdir(parents=True, exist_ok=True)
    source = verify_pretrained_checkpoint()
    config = run_config("FCMAE_P1_preflight", manifest_meta, train_rows, validation_rows, test_rows)
    config["pretrained_source"] = source
    write_json(preflight_root / "config.json", config)
    backbone = build_backbone(pretrained=True)
    mean, std = backbone.pretrained_cfg["mean"], backbone.pretrained_cfg["std"]
    encoder = FCMAEEncoder(backbone).to(DEVICE)
    decoder = FCMAEDecoder().to(DEVICE)
    modules = {"encoder": encoder, "fcmae_decoder": decoder}
    parameters = [p for module in modules.values() for p in module.parameters()]
    optimizer = torch.optim.AdamW(parameters, lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
    scaler = make_scaler()
    train_set = AnchorDataset(train_rows, training=True)
    sampler = EpochUniformReplacementSampler(train_set, "p1_preflight")
    loader = DataLoader(train_set, batch_size=MICRO_BATCH, sampler=sampler, num_workers=NUM_WORKERS)
    if len(loader) != ACCUM_STEPS: raise RuntimeError("preflight must contain exactly eight micro-batches")
    encoder.train(); decoder.train(); optimizer.zero_grad(set_to_none=True)
    losses = []
    for index, batch in enumerate(loader):
        generator = make_generator(stable_seed(SEED, FOLD, "p1_preflight_mask", index))
        with amp_context(): loss, _ = p1_forward(encoder, decoder, batch["image"], mean, std, generator)
        if not torch.isfinite(loss): raise FloatingPointError("non-finite preflight loss")
        scaler.scale(loss / ACCUM_STEPS).backward(); losses.append(float(loss.detach()))
    scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(parameters, 1.0)
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    trained_state = {f"{name}.{key}": value for name, module in modules.items() for key, value in module.state_dict().items()}
    trained_hash = tensor_state_sha256(trained_state)
    temporary = preflight_root / "reload_check.tmp.pth"
    torch.save({"modules": {name: module.state_dict() for name, module in modules.items()}, "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict()}, temporary)
    reloaded_encoder = FCMAEEncoder(build_backbone(pretrained=False)).to(DEVICE)
    reloaded_decoder = FCMAEDecoder().to(DEVICE)
    payload = torch.load(temporary, map_location=DEVICE, weights_only=False)
    strict_load(reloaded_encoder, payload["modules"]["encoder"], "preflight encoder")
    strict_load(reloaded_decoder, payload["modules"]["fcmae_decoder"], "preflight decoder")
    reloaded_state = {f"encoder.{key}": value for key, value in reloaded_encoder.state_dict().items()}
    reloaded_state.update({f"fcmae_decoder.{key}": value for key, value in reloaded_decoder.state_dict().items()})
    if tensor_state_sha256(reloaded_state) != trained_hash: raise RuntimeError("preflight strict-reload hash mismatch")
    temporary.unlink()
    validation_loader = DataLoader(AnchorDataset(validation_rows, training=False), batch_size=MICRO_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    batch = next(iter(validation_loader)); reloaded_encoder.eval(); reloaded_decoder.eval()
    with torch.no_grad(), amp_context():
        validation_loss, feature = p1_forward(reloaded_encoder, reloaded_decoder, batch["image"], mean, std, make_generator(stable_seed(SEED, FOLD, "preflight_validation")))
    feature_std = float(feature.float().std())
    if not torch.isfinite(validation_loss) or not np.isfinite(feature_std) or feature_std <= FEATURE_STD_MIN: raise RuntimeError("invalid preflight validation/features")
    summary = {"status": "PASS", "optimizer_updates": 1, "draws": len(sampler), "micro_batches": len(loader), "train_loss": float(np.mean(losses)), "validation_loss": float(validation_loss), "feature_std": feature_std, "post_update_parameter_sha256": trained_hash, "resource_usage": resource_usage(started), "weights_discarded": True}
    write_json(preflight_root / "summary.json", summary)
    return summary

In [ ]:
if RUN_REAL_DATA:
    ready_rows, train_rows, validation_rows, test_rows, manifest_meta = load_certified_folds(FOLD)
    print({"train": len(train_rows), "validation": len(validation_rows), "test_not_opened": len(test_rows)})
    try:
        if RUN_PREFLIGHT:
            print(run_preflight(train_rows, validation_rows, test_rows, manifest_meta))
        if RUN_P1:
            print(train_p1(train_rows, validation_rows, test_rows, manifest_meta))
        if RUN_P2:
            print(train_p2(train_rows, validation_rows, test_rows, manifest_meta))
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            "FCMAE OOM: restart the attempt from the pinned initialization with half MICRO_BATCH "
            "and double ACCUM_STEPS; do not resume an incompatible runtime configuration."
        ) from error
else:
    print("DATA-FREE MODE: definitions loaded. Set RUN_REAL_DATA=True only from an approved Stage 2 task brief.")